# DTW Analysis (Treatment-based)


Analysis of daily DTW incidents aggregated at **treatment** level (`2026-07-20-dtw_incident_daily_v2_treatment_based.csv`), inspired by `2026-07-20-incidend-analysis.ipynb`.

Key differences vs signal-based notebook:
- Entity key is `treatment_id` (not `audience_id`)
- Extra fields: `spend_total_7d`, `meets_pylon_spend_threshold`, `n_signals`
- Pylon merge uses `treatment_id` ↔ `treatment_id`


In [6]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts

In [7]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from Analysis.DTW_Incidents.dtw_functions import (
    annotate_pylon_unmatched_reasons,
    combine_incidents,
    merge_dtw_daily_with_pylon,
)
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.call_api_with_account_id import call_api_with_accountId

logger = logging.getLogger(__name__)
sns.set_theme(style="whitegrid", context="notebook")


In [31]:
workspace_ids = {ws["name"]: ws["id"] for ws in return_workspace_ids()}
dtw_path = "Analysis/DTW_Incidents/2026-07-20-dtw_incident_daily_v2_treatment_based.csv"
pylon_path = "Analysis/DTW_Incidents/2026-07-20_pylon_tickets_dtw_only.csv"
url = "https://targeting.innkeepr.ai/api/"
path_to_analysis = "Analysis/DTW_Incidents/2026-07-22-analyse/"

# Load data


In [9]:
dtw_daily = pd.read_csv(dtw_path)
dtw_daily["date"] = pd.to_datetime(dtw_daily["date"])
dtw_daily["is_incident"] = dtw_daily["is_incident"].astype(bool)
dtw_daily["meets_pylon_spend_threshold"] = dtw_daily["meets_pylon_spend_threshold"].astype(bool)
dtw_daily["week"] = dtw_daily["date"].dt.isocalendar().week.astype(int)
dtw_daily["year"] = dtw_daily["date"].dt.isocalendar().year.astype(int)
dtw_daily["month"] = dtw_daily["date"].dt.month.astype(int)
dtw_daily["year-week"] = (
    dtw_daily["year"].astype(str) + "-" + dtw_daily["week"].astype(str).str.zfill(2)
)
dtw_daily["year-month"] = (
    dtw_daily["year"].astype(str) + "-" + dtw_daily["month"].astype(str).str.zfill(2)
)

print(f"rows: {len(dtw_daily)}")
print(f"treatments: {dtw_daily['treatment_id'].nunique()}")
print(f"accounts: {dtw_daily['account_name'].nunique()}")
print(f"date range: {dtw_daily['date'].min().date()} → {dtw_daily['date'].max().date()}")
print(dtw_daily["model_type"].value_counts().to_string())
dtw_daily.head()


In [10]:
number_workspaces = dtw_daily["account_name"].unique()
print(f"Number of workspaces: {len(number_workspaces)}")
sorted(number_workspaces)

# Preprocess Data


In [11]:
# Map workspace ids
dtw_daily["workspace_id"] = dtw_daily["account_name"].map(workspace_ids)
print("missing workspace_id:", dtw_daily["workspace_id"].isna().sum())
dtw_daily.loc[dtw_daily["workspace_id"].isna(), "account_name"].unique()


In [125]:
# Infer ad platform / connection from treatment_resource (no API call)
def infer_connection(treatment_id: str, workspace_id: str) -> str:
    url_treatment = f"{url}treatments/query"
    url_connection = f"{url}connections/query"
    content = {"id":treatment_id}
    try:    
        treatment_def = call_api_with_accountId(url_treatment, workspace_id, content, logger)
        connection = treatment_def[0].get("connection", None)
        if connection is None:
            raise ValueError("Stop here - Connection is missing")
    except Exception as e:
        print("content = ", content)
        if "Provided invalid workspaceId" in str(e):
            return None
        raise Exception(e)
    connection_def = call_api_with_accountId(url_connection, workspace_id, {"_id": connection}, logger)
    connection = connection_def[0].get("name", None)
    if connection is None:
        raise ValueError("Stop here - Connection name is missing")
    return connection

In [95]:

treatments = dtw_daily[["treatment_id", "workspace_id"]].drop_duplicates()
print(f"Number of treatments: {len(treatments)}")
treatments["connection"] = treatments[["treatment_id", "workspace_id"]].apply(lambda x: infer_connection(x["treatment_id"], x["workspace_id"]), axis=1)
treatments.head()

dtw_daily = pd.merge(dtw_daily, treatments, on=["treatment_id", "workspace_id"], how="left")
dtw_daily["connection"].value_counts(dropna=False)

## Merge DTW daily with Pylon tickets

Match on `treatment_id` ↔ `treatment_id`, `workspace_id`, and `pylon.created_at` vs `dtw_daily.date`
with priority: **exact** → **+1 day** → **-1 day**.


In [140]:
pylon = pd.read_csv(pylon_path)
dtw_daily_pylon_merged, pylon_unmatched = merge_dtw_daily_with_pylon(
    dtw_daily,
    pylon,
    match_on="treatment",
)

print(f"dtw_daily rows: {len(dtw_daily)}")
print(f"pylon tickets: {len(pylon)}")
print(f"dtw rows with pylon match: {dtw_daily_pylon_merged['issue_id'].notna().sum()}")
print(f"pylon tickets matched: {len(pylon) - len(pylon_unmatched)}")
print(f"pylon tickets unmatched: {len(pylon_unmatched)}")
if "pylon_date_match" in dtw_daily_pylon_merged.columns:
    print(dtw_daily_pylon_merged.loc[dtw_daily_pylon_merged["issue_id"].notna(), "pylon_date_match"].value_counts().to_string())
dtw_daily_pylon_merged.head()


### Pylon tickets which could not be merged


In [97]:
pylon_unmatched = annotate_pylon_unmatched_reasons(
    pylon_unmatched,
    dtw_daily_pylon_merged[["treatment_id", "workspace_id", "date"]].drop_duplicates(),
    match_on="treatment",
)

print("Unmatched by reason:")
print(pylon_unmatched["unmatched_reason"].value_counts(dropna=False).to_string())
print(f"Percentage of unmatched tickets: {len(pylon_unmatched) / len(pylon)}")
pylon_unmatched[
    ["issue_id", "number", "title", "state", "created_at", "treatment_id", "workspace_id", "unmatched_reason", "issue_url"]
].head(20)


In [100]:
def add_model_type(signal_id: str, workspace_id: str, model_type: str) -> str:
    if type(signal_id) is float:
        if np.isnan(signal_id):
            return model_type
    url_signals = f"{url}signals/query"
    url_models = f"{url}models/query"
    content = {"id":signal_id}
    try:
        signal_def = call_api_with_accountId(url_signals, workspace_id, content, logger)
        model = signal_def[0].get("model", None)
        if model is None:
            raise ValueError("Stop here - Model is missing")
    except Exception as e:
        if "Provided invalid workspaceId" in str(e):
            print(f"For signal {signal_id}, workspace_id {workspace_id} is invalid")
            return model_type
        print(f"signal_id = {signal_id}, content = {content}")
        raise Exception(e)
    try:
        model_def = call_api_with_accountId(url_models, workspace_id, {"id": model}, logger)
        model_type = model_def[0].get("type", None)
    except Exception as e:
        print(f" model = {model}")
        raise Exception(e)
    if model_type is None:
        raise ValueError("Stop here - Model type is missing")
    return model_type

In [ ]:
def resolve_connection(row) -> str:
    conn = row["connection"]
    treatment_id = row["treatment_id"]
    workspace_id = row["workspace_id"]
    if pd.isna(treatment_id):
        return conn
    if conn != "unknown":
        return conn
    return infer_connection(treatment_id, workspace_id)

In [ ]:
dtw_daily_pylon = pd.concat([dtw_daily_pylon_merged, pylon_unmatched], ignore_index=True)
dtw_daily_pylon["connection"] = dtw_daily_pylon["connection"].fillna("unknown")
dtw_daily_pylon["connection"].value_counts(dropna=False)
dtw_daily_pylon["connection"] = dtw_daily_pylon[
    ["connection", "treatment_id", "workspace_id"]
].apply(resolve_connection, axis=1)

dtw_daily_pylon["connection"].value_counts(dropna=False)
print(dtw_daily_pylon["connection"].value_counts(dropna=False))
# add model type
dtw_daily_pylon["model_type"] = dtw_daily_pylon[["signal_id", "workspace_id","model_type"]].apply(lambda x: add_model_type(x["signal_id"], x["workspace_id"], x["model_type"]) if x["model_type"] in [None, np.nan] else x["model_type"], axis=1)
print(dtw_daily_pylon["model_type"].value_counts(dropna=False))
print(dtw_daily_pylon["connection"].value_counts(dropna=False))

# Analysis


## Overview: model type, connection, spend threshold


In [130]:
model_type_incidents = dtw_daily_pylon.groupby(by=["model_type", "connection","is_incident"]).agg(
    count=("is_incident", "count"),
).reset_index()
model_type_incidents = model_type_incidents.pivot(index=["model_type", "connection"], columns="is_incident", values="count")
model_type_incidents["ratio"] = model_type_incidents[True] / (model_type_incidents[False]+model_type_incidents[True])
model_type_incidents.reset_index(inplace=True)#.drop(columns=["is_incident"])
model_type_incidents

## Weekly Incidents


In [131]:
weekly_incidents_workspaces = (
    dtw_daily_pylon.groupby(
        by=["account_name", "year", "week", "model_type"],
        as_index=False,
    )
    .agg(
        incident_days=("is_incident", "sum"),
        total_days=("is_incident", "count"),
        avg_spend_7d=("spend_total_7d", "mean"),
        treatments=("treatment_id", "nunique"),
    )
)
weekly_incidents_workspaces["incident_days"] = weekly_incidents_workspaces["incident_days"].astype(int)
weekly_incidents_workspaces["non_incident_days"] = (
    weekly_incidents_workspaces["total_days"] - weekly_incidents_workspaces["incident_days"]
)
weekly_incidents_workspaces["is_incident"] = (
    weekly_incidents_workspaces["incident_days"] / weekly_incidents_workspaces["total_days"]
)
weekly_incidents_workspaces["year-week"] = (
    weekly_incidents_workspaces["year"].astype(str)
    + "-"
    + weekly_incidents_workspaces["week"].astype(str).str.zfill(2)
)
weekly_incidents_workspaces = weekly_incidents_workspaces.sort_values(
    by=["year", "week"]
).reset_index(drop=True)
weekly_incidents_workspaces


### Overall weekly incident rates


In [132]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
plt.suptitle("Overall Incident Rates (Treatment-based)")
ax1.set_title("Incident Rate by Model Type")
sns.barplot(
    data=weekly_incidents_workspaces,
    x="year-week",
    y="is_incident",
    hue="model_type",
    ax=ax1,
    errorbar="sd",
)
ax1.set_ylabel("Incident rate")
ax1.set_ylim(-0.05, 1.05)
ax1.tick_params(axis="x", rotation=90)

ax2.set_title("Incident Days by Model Type")
sns.barplot(
    data=weekly_incidents_workspaces,
    x="year-week",
    y="incident_days",
    hue="model_type",
    ax=ax2,
    errorbar="sd",
)
ax2.set_ylabel("Incident days")
ax2.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()
fig.savefig(f"{path_to_analysis}/weekly_incidents_overall_model_type.png")

In [133]:
fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(
    data=weekly_incidents_workspaces,
    x="account_name",
    y="is_incident",
    hue="model_type",
    ax=ax,
)
ax.set_title("Weekly incident rate by account and model type (treatment-based)")
ax.set_xlabel("Account")
ax.set_ylabel("Incident rate")
ax.set_ylim(-0.05, 1.05)
ax.tick_params(axis="x", rotation=90)
ax.legend(title="Model type", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()


In [134]:
# Conversion-only pivot (accounts × weeks)
weekly_pivot_conversion = (
    weekly_incidents_workspaces[weekly_incidents_workspaces["model_type"] == "conversion"]
    .pivot(index="year-week", columns="account_name", values="is_incident")
)
weekly_pivot_conversion


### Weekly Pylon Tags


In [135]:
dtw_daily_pylon["tags"] = np.where(
    (dtw_daily_pylon["tags"].isna()) & (dtw_daily_pylon["issue_id"].notna()),
    "no_tags",
    dtw_daily_pylon["tags"],
)
dtw_daily_pylon["tags"] = np.where(
    (dtw_daily_pylon["tags"].isna()) & (dtw_daily_pylon["issue_id"].isna()),
    "no_pylon_ticket",
    dtw_daily_pylon["tags"],
)

weekly_tags = (
    dtw_daily_pylon.groupby(["year", "week", "model_type", "tags", "connection"], as_index=False)
    .agg(incident_days=("is_incident", "sum"))
)
weekly_tags["year-week"] = (
    weekly_tags["year"].astype(str) + "-" + weekly_tags["week"].astype(str).str.zfill(2)
)
weekly_tags["incident_days"] = weekly_tags["incident_days"].astype(int)
print(dtw_daily_pylon["tags"].value_counts(dropna=False).head(15).to_string())
weekly_tags.head()


In [143]:
def plot_tag_counts(plot_df, title):
    plot_df = plot_df.copy()
    plot_df["incident_days"] = plot_df["incident_days"].astype(int)
    connections = sorted(plot_df["connection"].dropna().unique())
    if not connections:
        print("No data to plot")
        return

    fig, axes = plt.subplots(len(connections), 1, figsize=(14, 4 * len(connections)), sharex=False)
    if len(connections) == 1:
        axes = [axes]

    for ax, conn in zip(axes, connections):
        sub = plot_df[plot_df["connection"] == conn]
        weeks = sorted(sub["year-week"].unique())
        tags = sorted(sub["tags"].unique())
        week_to_x = {w: i for i, w in enumerate(weeks)}
        tag_to_y = {t: i for i, t in enumerate(tags)}
        min_c, max_c = sub["incident_days"].min(), sub["incident_days"].max()

        ax.set_yticks(range(len(tags)))
        ax.set_yticklabels(tags)
        ax.set_xticks(range(len(weeks)))
        ax.set_xticklabels(weeks, rotation=90)
        ax.set_xlim(-0.5, len(weeks) - 0.5)
        ax.set_ylim(-0.5, len(tags) - 0.5)
        ax.grid(True, alpha=0.3)
        ax.set_title(f"{title} — {conn}")

        for _, row in sub.iterrows():
            fontsize = 12 if max_c == min_c else 9 + 9 * (row["incident_days"] - min_c) / (max_c - min_c)
            ax.text(
                week_to_x[row["year-week"]],
                tag_to_y[row["tags"]],
                str(int(row["incident_days"])),
                ha="center",
                va="center",
                fontsize=fontsize,
                fontweight="bold",
                color="C0",
            )

    plt.tight_layout()
    plt.show()
    return fig


plot_df_conversion = weekly_tags[
    (weekly_tags["tags"] != "no_pylon_ticket")
    & (weekly_tags["model_type"] == "conversion")
]
fig = plot_tag_counts(plot_df_conversion, "Conversion tag counts over time")
fig.savefig(f"{path_to_analysis}/weekly_tags_conversion.png")


In [144]:
plot_df_causal = weekly_tags[
    (weekly_tags["tags"] != "no_pylon_ticket")
    & (weekly_tags["model_type"] == "causal")
]
fig = plot_tag_counts(plot_df_causal, "Causal tag counts over time")
fig.savefig(f"{path_to_analysis}/weekly_tags_causal.png")


## Incidents by Month


In [145]:
monthly_incidents_workspaces = (
    dtw_daily_pylon.groupby(
        by=["account_name", "year", "month", "model_type", "connection"],
        as_index=False,
    )
    .agg(
        incident_days=("is_incident", "sum"),
        total_days=("is_incident", "count"),
        avg_spend_7d=("spend_total_7d", "mean"),
    )
)
monthly_incidents_workspaces["is_incident"] = (
    monthly_incidents_workspaces["incident_days"] / monthly_incidents_workspaces["total_days"]
)
monthly_incidents_workspaces["year-month"] = (
    monthly_incidents_workspaces["year"].astype(str)
    + "-"
    + monthly_incidents_workspaces["month"].astype(str).str.zfill(2)
)
monthly_incidents_workspaces["model_type-connection"] = (
    monthly_incidents_workspaces["model_type"].astype(str)
    + "-"
    + monthly_incidents_workspaces["connection"].astype(str).str.zfill(2)
)
monthly_incidents_workspaces = monthly_incidents_workspaces.sort_values(["year", "month"]).reset_index(drop=True)
monthly_incidents_workspaces.groupby(["year-month", "model_type"])["is_incident"].describe().sort_values(by=["model_type","year-month"])


In [24]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
plt.suptitle("Overall Incident Rates per Month (Treatment-based)")
sns.barplot(
    data=monthly_incidents_workspaces,
    x="year-month",
    y="is_incident",
    hue="model_type",
    ax=ax1,
    errorbar="sd",
)
ax1.set_title("Incident Rate by Model Type")
ax1.set_ylim(-0.05, 1.05)

sns.barplot(
    data=monthly_incidents_workspaces,
    x="year-month",
    y="incident_days",
    hue="model_type",
    ax=ax2,
    errorbar="sd",
)
ax2.set_title("Incident Days by Model Type")
plt.tight_layout()
plt.show()


In [147]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
monthly_incidents_workspaces = monthly_incidents_workspaces.sort_values(by=["year-month","connection"])
plt.suptitle("Overall Incident Rates per Month and Connection")
ax1.set_title("Incident Rate by Connection for conversion models")
sns.barplot(data=monthly_incidents_workspaces[monthly_incidents_workspaces["model_type"]=="conversion"], x="year-month", y="is_incident", hue="model_type-connection", ax=ax1, errorbar="sd")
plt.grid(True)
plt.legend(title="Connection", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
ax2.set_title("Incident Rate by Connection for causal models")
sns.barplot(data=monthly_incidents_workspaces[monthly_incidents_workspaces["model_type"]=="causal"], x="year-month", y="is_incident", hue="model_type-connection", ax=ax2, errorbar="sd")
plt.grid(True)
plt.legend(title="Connection", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
fig.savefig(f"{path_to_analysis}/monthly_incidents_workspaces_connection.png")

## Incidents by Definition

- Combine incidents per `account_name` + `treatment_id`
- `start_date`: first incident after a false
- `end_date`: last true before 3 consecutive false days
- Also emits `no_incident` gaps


In [148]:
incidents_by_definition = combine_incidents(
    dtw_daily_pylon,
    entity_id_col="treatment_id",
)
incidents_by_definition = incidents_by_definition.sort_values(
    by=["account_name", "treatment_id", "start_date"]
)
incidents_by_definition["label"] = (
    incidents_by_definition["model_type"].astype(str)
    + " "
    + incidents_by_definition["period_type"].astype(str)
)
print(incidents_by_definition["period_type"].value_counts().to_string())
incidents_by_definition.head(20)


In [149]:
duration_stats = incidents_by_definition.groupby("label")["duration_days"].describe()
duration_stats


In [150]:
fig, ax = plt.subplots(figsize=(14, 6))
sns.histplot(
    data=incidents_by_definition,
    x="duration_days",
    hue="label",
    ax=ax,
    kde=True,
    element="step",
    stat="density",
    common_norm=False,
)
ax.set_title("Incident / no-incident duration distribution (treatment-based)")
ax.set_xlabel("Duration (days)")
plt.tight_layout()
plt.show()
fig.savefig(f"{path_to_analysis}/incidents_by_definition_duration.png")
